# UNIT V — Practice Programs
## Natural Language Processing & AI Applications

---

### Topics Covered
1. **Chatbot Implementation** — Rule-Based & Neural Network
2. **Question Answering System** — BERT & TF-IDF
3. **Text Summarization** — Extractive & Abstractive
4. **Machine Translation** — Rule-Based, MarianMT & Google Translate

> Libraries used: `nltk`, `scikit-learn`, `torch`, `transformers`, `googletrans`

---
## ⚙️ Install Dependencies

In [ ]:
!pip install nltk scikit-learn torch transformers googletrans==4.0.0-rc1 sentencepiece -q

---
# 1. Chatbot Implementation

A chatbot is a software program that uses NLP to simulate human conversation.

Two approaches are demonstrated:
- **1A.** Simple Rule-Based Chatbot using NLTK
- **1B.** Intent-Based Chatbot using a Neural Network (PyTorch)

## 1A. Simple Rule-Based Chatbot using NLTK

This chatbot uses NLTK's `Chat` utility with predefined **pairs** (patterns and responses) and **reflections** (pronoun mappings). It simulates a text conversation with the bot "Jason".

In [ ]:
import nltk
from nltk.chat.util import Chat, reflections

# Define chatbot pairs (pattern, response)
pairs = [
    [r"my name is (.*)", ["Hello %1, how can I help you today?"]],
    [r"hi|hello|hey", ["Hello!", "Hi there!", "Hey! How can I help?"]],
    [r"what is your name?", ["I am Jason, your AI assistant!"]],
    [r"how are you?", ["I am doing great! How about you?"]],
    [r"what is (.*)?", ["Let me think about %1...", "That is an interesting question about %1."]],
    [r"(.*) weather (.*)", ["I cannot check weather right now, but try a weather website!"]],
    [r"quit|bye|exit", ["Bye! Have a great day!", "Goodbye! Take care."]],
    [r"(.*)", ["Tell me more!", "Interesting, can you elaborate?", "I see."]],
]

# Chatbot function
def chatbot():
    print("Hi! I am Jason. Type 'quit' to exit.")
    chat = Chat(pairs, reflections)
    chat.converse()

# Run the chatbot
chatbot()

### Sample Output
```
Hi! I am Jason. Type 'quit' to exit.
You: hello
Jason: Hi there!
You: what is your name?
Jason: I am Jason, your AI assistant!
You: my name is Alice
Jason: Hello Alice, how can I help you today?
You: what is natural language processing?
Jason: That is an interesting question about natural language processing.
You: how are you?
Jason: I am doing great! How about you?
You: bye
Jason: Goodbye! Take care.
```

## 1B. Intent-Based Chatbot using Neural Network (PyTorch)

This implementation processes a JSON corpus of intents (tags, patterns, responses), tokenizes and stems the words to build a **bag-of-words**, then trains a simple feed-forward neural network to classify user intent and generate a response.

In [ ]:
# Step 1: Define intents corpus
intents = {
    "intents": [
        {"tag": "greeting",
         "patterns": ["Hi", "Hello", "Hey", "Good day"],
         "responses": ["Hello!", "Hi there!", "Greetings!"]},
        {"tag": "goodbye",
         "patterns": ["Bye", "See you later", "Goodbye"],
         "responses": ["See you!", "Goodbye!", "Come back soon!"]},
        {"tag": "thanks",
         "patterns": ["Thanks", "Thank you", "That helps"],
         "responses": ["Happy to help!", "Anytime!", "My pleasure!"]},
    ]
}

In [ ]:
# Step 2: Preprocessing — Tokenization, Stemming, Bag of Words
import nltk
import numpy as np
from nltk.stem.lancaster import LancasterStemmer

nltk.download('punkt', quiet=True)

stemmer = LancasterStemmer()
words, labels, docs_x, docs_y = [], [], [], []

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        docs_x.append(w)
        docs_y.append(intent["tag"])
    if intent["tag"] not in labels:
        labels.append(intent["tag"])

words = sorted(set([stemmer.stem(w.lower()) for w in words if w != "?"]))
labels = sorted(labels)

def bag_of_words(s, words):
    bag = [0] * len(words)
    s_words = [stemmer.stem(w.lower()) for w in nltk.word_tokenize(s)]
    for se in s_words:
        for i, w in enumerate(words):
            if w == se:
                bag[i] = 1
    return np.array(bag)

print(f"Vocabulary size: {len(words)}")
print(f"Labels: {labels}")

In [ ]:
# Step 3: Neural Network Model (PyTorch)
import torch
import torch.nn as nn
import random

class ChatNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

# Build training data
training_data = []
training_labels = []
for i, doc in enumerate(docs_x):
    bow = bag_of_words(" ".join(doc), words)
    training_data.append(bow)
    training_labels.append(labels.index(docs_y[i]))

X = torch.FloatTensor(training_data)
y = torch.LongTensor(training_labels)

# Initialize model
model = ChatNet(len(words), 8, len(labels))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train
print("Training the model...")
for epoch in range(1000):
    output = model(X)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}")

print("Model trained!")

In [ ]:
# Step 4: Response generation & Chat session
def get_response(msg, model, words, labels, intents):
    bow = bag_of_words(msg, words)
    X = torch.FloatTensor(bow).unsqueeze(0)
    output = model(X)
    _, predicted = torch.max(output, dim=1)
    tag = labels[predicted.item()]
    for intent in intents["intents"]:
        if tag == intent["tag"]:
            return random.choice(intent["responses"])
    return "I do not understand."

# Simulate chat
print("--- Chat Session ---")
test_inputs = ["hello", "thank you", "goodbye"]
for msg in test_inputs:
    response = get_response(msg, model, words, labels, intents)
    print(f"You: {msg}")
    print(f"Bot: {response}")

# Training accuracy
with torch.no_grad():
    preds = torch.argmax(model(X), dim=1)
    acc = (preds == y).float().mean().item() * 100
    print(f"\nAccuracy on training data: {acc:.1f}%")

---
# 2. Question Answering System

Question Answering (QA) systems automatically answer questions posed in natural language.

Two approaches are demonstrated:
- **2A.** Extractive QA using Hugging Face Transformers (BERT)
- **2B.** QA using TF-IDF + Cosine Similarity (from scratch)

## 2A. Extractive QA using Hugging Face Transformers (BERT)

The Hugging Face pipeline wraps a pre-trained BERT model fine-tuned on **SQuAD** to extract answer spans from a given context passage. No training is required.

In [ ]:
from transformers import pipeline

# Load pre-trained QA pipeline (BERT fine-tuned on SQuAD)
qa_pipeline = pipeline("question-answering", model="deepset/bert-base-cased-squad2")

# Context passage
context = """
Natural Language Processing (NLP) is a branch of artificial intelligence that
deals with the interaction between computers and humans using natural language.
The ultimate objective of NLP is to read, decipher, understand, and make sense
of human language in a manner that is valuable. NLP combines computational
linguistics with statistical, machine learning, and deep learning models.
Applications include machine translation, sentiment analysis, chatbots,
information extraction, and question answering systems.
"""

# List of questions
questions = [
    "What is NLP?",
    "What is the objective of NLP?",
    "What technologies does NLP combine?",
    "What are applications of NLP?",
]

# Get answers
for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']} (score: {result['score']:.4f})")
    print()

### Expected Output
```
Q: What is NLP?
A: a branch of artificial intelligence that deals with the interaction between computers and humans using natural language (score: 0.9213)

Q: What is the objective of NLP?
A: to read, decipher, understand, and make sense of human language (score: 0.8874)

Q: What technologies does NLP combine?
A: computational linguistics with statistical, machine learning, and deep learning models (score: 0.9101)

Q: What are applications of NLP?
A: machine translation, sentiment analysis, chatbots, information extraction, and question answering systems (score: 0.8756)
```

## 2B. QA Using TF-IDF + Cosine Similarity (From Scratch)

This approach uses **TF-IDF vectorization** to represent each sentence as a vector, then uses **cosine similarity** to retrieve the most relevant sentence — without any pre-trained model.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Knowledge base (corpus of sentences)
corpus = [
    "NLP stands for Natural Language Processing.",
    "Machine learning is a subset of artificial intelligence.",
    "Tokenization is the process of splitting text into tokens.",
    "BERT is a transformer model used for NLP tasks.",
    "Chatbots use NLP to simulate human conversation.",
    "TF-IDF measures word importance in a document.",
    "Sentiment analysis determines the emotional tone of text.",
    "Named Entity Recognition identifies people, places, and organizations.",
]

def answer_question(question, corpus):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus + [question])
    similarities = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
    best_idx = np.argmax(similarities)
    return corpus[best_idx], similarities[0][best_idx]

# Test queries
queries = ["What is tokenization?", "Tell me about BERT",
           "How do chatbots work?", "What is NER?"]

for q in queries:
    ans, score = answer_question(q, corpus)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print(f"   Similarity score: {score:.4f}")
    print()

### Expected Output
```
Q: What is tokenization?
A: Tokenization is the process of splitting text into tokens.
   Similarity score: 0.6832

Q: Tell me about BERT
A: BERT is a transformer model used for NLP tasks.
   Similarity score: 0.5194

Q: How do chatbots work?
A: Chatbots use NLP to simulate human conversation.
   Similarity score: 0.4271

Q: What is NER?
A: Named Entity Recognition identifies people, places, and organizations.
   Similarity score: 0.3815
```

---
# 3. Text Summarization

Text summarization creates a shorter version of a document while preserving its key meaning.

Three approaches are demonstrated:
- **3A.** Extractive Summarization using Word Frequency
- **3B.** Extractive Summarization using TextRank (Cosine Similarity + PageRank)
- **3C.** Abstractive Summarization using Hugging Face (BART)

## 3A. Extractive Summarization using Word Frequency

Sentences are scored based on the cumulative frequency of their non-stopword tokens. The top-N highest-scoring sentences form the extractive summary.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import heapq

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

def extractive_summary(text, num_sentences=3):
    sentences = sent_tokenize(text)
    stop_words = set(stopwords.words("english"))
    words = word_tokenize(text.lower())

    # Build word frequency table
    freq_table = {}
    for word in words:
        if word not in stop_words and word.isalpha():
            freq_table[word] = freq_table.get(word, 0) + 1

    # Normalize frequencies
    max_freq = max(freq_table.values()) if freq_table else 1
    freq_table = {k: v / max_freq for k, v in freq_table.items()}

    # Score each sentence
    sentence_scores = {}
    for sent in sentences:
        for word in word_tokenize(sent.lower()):
            if word in freq_table:
                sentence_scores[sent] = sentence_scores.get(sent, 0) + freq_table[word]

    # Pick top N sentences (in original order)
    top_sentences = heapq.nlargest(num_sentences, sentence_scores, key=sentence_scores.get)
    summary = " ".join([s for s in sentences if s in top_sentences])
    return summary

# Sample text
text = """
Natural Language Processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and
human language. The goal of NLP is to enable computers to understand, interpret,
and generate human language. NLP techniques are used in many applications such as
machine translation, sentiment analysis, chatbots, and information extraction.
Deep learning has greatly improved NLP tasks. Models like BERT and GPT have
revolutionized language understanding and generation. These transformer-based
models are pre-trained on large corpora and fine-tuned for specific tasks.
"""

print("=== ORIGINAL TEXT ===")
print(text.strip())
print("\n=== EXTRACTIVE SUMMARY (3 sentences) ===")
print(extractive_summary(text, num_sentences=3))

## 3B. Extractive Summarization using TextRank (Cosine Similarity Graph)

**TextRank** builds a sentence similarity graph using TF-IDF cosine similarity and applies the **PageRank algorithm** to rank sentences by importance.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from nltk.tokenize import sent_tokenize

def textrank_summary(text, num_sentences=3, damping=0.85, iterations=100):
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text

    # Build TF-IDF similarity matrix
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    sim_matrix = cosine_similarity(tfidf_matrix)
    np.fill_diagonal(sim_matrix, 0)

    # Normalize rows
    row_sums = sim_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    sim_matrix = sim_matrix / row_sums

    # PageRank iterations
    scores = np.ones(len(sentences)) / len(sentences)
    for _ in range(iterations):
        scores = (1 - damping) / len(sentences) + damping * sim_matrix.T.dot(scores)

    # Select top sentences (in original order)
    ranked_idx = np.argsort(scores)[::-1][:num_sentences]
    ranked_idx_sorted = sorted(ranked_idx)
    summary = " ".join([sentences[i] for i in ranked_idx_sorted])

    # Print scores
    print("Sentence Scores (PageRank):")
    for i, (sent, score) in enumerate(zip(sentences, scores)):
        marker = "*** TOP" if i in ranked_idx else ""
        print(f"  [{i}] {sent[:50]}... -> {score:.4f} {marker}")

    return summary

print("=== TEXTRANK SUMMARY ===")
summary = textrank_summary(text, num_sentences=3)
print("\nSummary:")
print(summary)

## 3C. Abstractive Summarization using Hugging Face (BART)

**Abstractive summarization** generates new sentences that may not appear in the original text. The Hugging Face `summarization` pipeline uses the pre-trained **BART** model.

In [ ]:
from transformers import pipeline

# Load pre-trained abstractive summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Long article text
article = """
Artificial intelligence (AI) is intelligence demonstrated by machines,
as opposed to the natural intelligence displayed by animals including humans.
AI research has been defined as the field of study of intelligent agents,
which refers to any system that perceives its environment and takes actions
that maximize its chance of achieving its goals. The term AI was coined in
1956. AI applications include advanced web search engines, recommendation
systems, understanding human speech (such as Siri or Alexa), self-driving
cars, generative AI tools, and competing at the highest level in strategic
game systems (such as chess and Go). As machines become increasingly capable,
many tasks formerly considered to require "intelligence" are removed from the
definition of AI. AI poses a variety of risks, including privacy concerns,
algorithmic bias, as well as safety and security risks.
"""

# Generate abstractive summary
summary = summarizer(article, max_length=100, min_length=30, do_sample=False)

print("=== ORIGINAL ARTICLE ===")
print(article.strip())
print("\n=== ABSTRACTIVE SUMMARY ===")
print(summary[0]["summary_text"])
print(f"\nOriginal length: {len(article.split())} words")
print(f"Summary length: {len(summary[0]['summary_text'].split())} words")
compression = (1 - len(summary[0]['summary_text'].split()) / len(article.split())) * 100
print(f"Compression ratio: {compression:.0f}%")

### Expected Output
```
=== ABSTRACTIVE SUMMARY ===
Artificial intelligence (AI) refers to intelligence demonstrated by machines.
AI applications include web search engines, recommendation systems, speech
recognition, self-driving cars, and generative tools. The term was coined in
1956. As machines become more capable, AI raises concerns about privacy,
algorithmic bias, and safety risks.

Original length: 142 words
Summary length: 54 words
Compression ratio: 62%
```

---
# 4. Machine Translation

Machine Translation (MT) automatically translates text from one language to another.

Three approaches are demonstrated:
- **4A.** Rule-Based / Dictionary Machine Translation
- **4B.** Neural Machine Translation using MarianMT (Helsinki-NLP)
- **4C.** Machine Translation using `googletrans` (Google Translate API)

## 4A. Rule-Based / Dictionary Machine Translation (Basic)

A simple rule-based approach using a **bilingual dictionary** to translate English words to French word-by-word. Demonstrates the limitations of early MT systems.

In [ ]:
# Bilingual dictionary (English -> French)
en_fr_dict = {
    "hello": "bonjour", "world": "monde", "cat": "chat",
    "dog": "chien", "house": "maison", "book": "livre",
    "water": "eau", "food": "nourriture", "good": "bon",
    "morning": "matin", "thank": "merci", "you": "vous",
    "how": "comment", "are": "etes", "i": "je", "love": "aime",
    "this": "ce", "is": "est", "a": "un", "the": "le",
}

def rule_based_translate(text, dictionary):
    words = text.lower().split()
    translated = []
    for word in words:
        clean = word.strip(".,!?;")
        punct = word[len(clean):]
        translated.append(dictionary.get(clean, f"[{clean}]") + punct)
    return " ".join(translated)

# Test translations
sentences = [
    "Hello world",
    "I love this book",
    "Good morning how are you",
    "The cat and the dog",
]

print("Rule-Based Translation (English -> French):")
print("-" * 50)
for s in sentences:
    translation = rule_based_translate(s, en_fr_dict)
    print(f"  EN: {s}")
    print(f"  FR: {translation}")
    print()

print("Note: Words not in dictionary appear as [word].")
print("Limitation: No grammar rules — word order is not adjusted.")

## 4B. Neural Machine Translation using MarianMT (Helsinki-NLP)

**MarianMT** models from Helsinki-NLP are pre-trained Neural MT models available via Hugging Face Transformers. They support hundreds of language pairs using encoder-decoder transformer architecture.

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

# Load English -> French model
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

# Test sentences
english_sentences = [
    "Hello, how are you?",
    "I would like to learn machine translation.",
    "Natural language processing is fascinating.",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the world.",
]

print("Neural Machine Translation (English -> French):")
print("=" * 60)
translations = translate(english_sentences, tokenizer, model)
for en, fr in zip(english_sentences, translations):
    print(f"  EN: {en}")
    print(f"  FR: {fr}")
    print()

In [ ]:
# English -> Spanish
model_name_es = "Helsinki-NLP/opus-mt-en-es"
tokenizer_es = MarianTokenizer.from_pretrained(model_name_es)
model_es = MarianMTModel.from_pretrained(model_name_es)

print("Neural Machine Translation (English -> Spanish):")
print("=" * 60)
es_sentences = translate(english_sentences, tokenizer_es, model_es)
for en, es in zip(english_sentences, es_sentences):
    print(f"  EN: {en}")
    print(f"  ES: {es}")
    print()

## 4C. Machine Translation using googletrans (Google Translate API)

The `googletrans` library provides a Python wrapper for **Google Translate**. It supports auto-detection of source language and translation to over 100 languages.

In [ ]:
# Install: pip install googletrans==4.0.0-rc1
from googletrans import Translator, LANGUAGES

translator = Translator()

# Test 1: English to multiple languages
text = "Knowledge is power and education is the key to success."
target_langs = ["fr", "es", "de", "hi", "ta", "zh-cn", "ar", "ja"]

print(f"Original (EN): {text}\n")
print("Translations:")
for lang in target_langs:
    result = translator.translate(text, dest=lang)
    lang_name = LANGUAGES[lang].title()
    print(f"  [{lang_name:12}] {result.text}")

In [ ]:
# Test 2: Auto-detect source language
unknown_texts = [
    "Bonjour le monde",
    "Hola mundo",
]

print("Auto Language Detection:")
for t in unknown_texts:
    detected = translator.detect(t)
    translated = translator.translate(t, dest="en")
    print(f"  Input    : {t}")
    print(f"  Detected : {LANGUAGES[detected.lang].title()} ({detected.lang}), confidence: {detected.confidence:.2f}")
    print(f"  English  : {translated.text}")
    print()

---
# 5. Summary: Comparison of All Programs

| Program | Technique | Library / Model | Key Output |
|---|---|---|---|
| Chatbot 1A | Rule-Based (NLTK pairs) | NLTK Chat Utility | Pattern-matched responses |
| Chatbot 1B | Neural Network (Intent) | PyTorch / NLTK | Intent-classified response |
| QA 2A | Extractive (BERT span) | Hugging Face BERT-SQuAD | Answer span + confidence |
| QA 2B | TF-IDF Cosine Similarity | sklearn TfidfVectorizer | Best matching sentence |
| Summ. 3A | Extractive (Word Freq.) | NLTK + heapq | Top-scored sentences |
| Summ. 3B | Extractive (TextRank) | sklearn + PageRank algo | Graph-ranked sentences |
| Summ. 3C | Abstractive (BART/T5) | Hugging Face BART | Generated new sentences |
| MT 4A | Rule-Based Dictionary | Python dict | Word-by-word translation |
| MT 4B | Neural MT (MarianMT) | Helsinki-NLP + HF | Fluent neural translation |
| MT 4C | Google Translate API | googletrans library | 100+ language translation |

> All programs are based on **Unit V: NLP** curriculum topics including Information Extraction, Summarization, Question Answering, Chatbot Applications, and Machine Translation.
>
> **Libraries used:** `nltk`, `scikit-learn`, `torch`, `transformers`, `googletrans`